# 03 · CRUD en Cassandra

Keyspace `github_metrics`. Driver: `cassandra-driver` (CQL).

**Justificación:** escrituras masivas append-only y lecturas por rango de tiempo; cada tabla se diseña para una consulta.

In [1]:
import sys, os
sys.path.insert(0, "../src")          # modulos del proyecto
import config
from datetime import datetime, timezone, date
from pprint import pprint

In [2]:
s = config.cassandra_session()
for r in s.execute("SELECT table_name FROM system_schema.tables WHERE keyspace_name='github_metrics'"): print(r.table_name)

daily_totals
events_by_hour
kpis_by_date
pipeline_runs
repo_activity_by_day
user_activity_by_day
user_repo_interactions_by_hour


## C · Insertar

In [3]:
demo_day = date(2099, 1, 1)   # dia ficticio, no se mezcla con datos reales
s.execute('INSERT INTO events_by_hour (event_date, event_hour, event_type, event_count) VALUES (%s,%s,%s,%s)',
          (demo_day, 10, 'PushEvent', 123))
s.execute('INSERT INTO events_by_hour (event_date, event_hour, event_type, event_count) VALUES (%s,%s,%s,%s)',
          (demo_day, 11, 'PushEvent', 456))
print('insertado')

insertado


## R · Consultar

In [4]:
rows = s.execute('SELECT * FROM events_by_hour WHERE event_date=%s', (demo_day,))
for r in rows: print(r)

Row(event_date=Date(47117), event_hour=10, event_type='PushEvent', event_count=123)
Row(event_date=Date(47117), event_hour=11, event_type='PushEvent', event_count=456)


In [5]:
# eventos por hora del ultimo dia cargado
last = s.execute('SELECT event_date FROM daily_totals WHERE year=%s LIMIT 1', (date.today().year,)).one()
if last:
    d = last.event_date.date()
    print('día:', d)
    for r in s.execute('SELECT event_hour, event_type, event_count FROM events_by_hour WHERE event_date=%s LIMIT 10', (d,)): print(r)

día: 2026-11-10
Row(event_hour=0, event_type='CommitCommentEvent', event_count=8)
Row(event_hour=0, event_type='ForkEvent', event_count=43)
Row(event_hour=0, event_type='GollumEvent', event_count=2)
Row(event_hour=0, event_type='IssueCommentEvent', event_count=598)
Row(event_hour=0, event_type='IssuesEvent', event_count=495)
Row(event_hour=0, event_type='MemberEvent', event_count=15)
Row(event_hour=0, event_type='PublicEvent', event_count=5)
Row(event_hour=0, event_type='PullRequestEvent', event_count=1236)
Row(event_hour=0, event_type='PullRequestReviewCommentEvent', event_count=64)
Row(event_hour=0, event_type='PullRequestReviewEvent', event_count=333)


In [6]:
# kpis calculados por el dag
if last:
    for r in s.execute('SELECT kpi_name, kpi_value, kpi_label FROM kpis_by_date WHERE event_date=%s', (d,)): print(r)

Row(kpi_name='avg_events_per_user', kpi_value=1.9, kpi_label='33698 usuarios')
Row(kpi_name='daily_growth_pct', kpi_value=-20.9, kpi_label='vs 2026-11-09 (horas 0–18)')
Row(kpi_name='events_per_day', kpi_value=64019.0, kpi_label='eventos')
Row(kpi_name='pr_merge_rate_pct', kpi_value=95.05, kpi_label='9308 merged / 485 cerrados sin fusionar')
Row(kpi_name='top_repo_connections', kpi_value=121.0, kpi_label='morluto/rea')


## U · Actualizar

In [7]:
s.execute('UPDATE events_by_hour SET event_count=%s WHERE event_date=%s AND event_hour=%s AND event_type=%s',
          (999, demo_day, 10, 'PushEvent'))
print(s.execute('SELECT * FROM events_by_hour WHERE event_date=%s AND event_hour=%s', (demo_day, 10)).one())

Row(event_date=Date(47117), event_hour=10, event_type='PushEvent', event_count=999)


## D · Eliminar

In [8]:
s.execute('DELETE FROM events_by_hour WHERE event_date=%s AND event_hour=%s AND event_type=%s', (demo_day, 10, 'PushEvent'))
s.execute('DELETE FROM events_by_hour WHERE event_date=%s', (demo_day,))   # borra la particion completa
print('quedan:', list(s.execute('SELECT * FROM events_by_hour WHERE event_date=%s', (demo_day,))))

quedan: []
